In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Load raw data
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Fix TotalCharges — convert string to float
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check how many NaN were created
print("NaN in TotalCharges after conversion:", df['TotalCharges'].isnull().sum())

# Fill NaN with 0 (new customers with no charges yet)
df['TotalCharges'].fillna(0, inplace=True)

# Confirm fix
print("TotalCharges dtype now:", df['TotalCharges'].dtype)
print("Any remaining NaN:", df['TotalCharges'].isnull().sum())

NaN in TotalCharges after conversion: 11
TotalCharges dtype now: float64
Any remaining NaN: 11


C:\Users\nandi\AppData\Local\Temp\ipykernel_17660\1579540445.py:15: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['TotalCharges'].fillna(0, inplace=True)


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Load raw data
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Fix TotalCharges — convert string to float
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check how many NaN were created
print("NaN after conversion:", df['TotalCharges'].isnull().sum())

# Correct way to fill NaN
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Confirm fix
print("TotalCharges dtype:", df['TotalCharges'].dtype)
print("NaN remaining:", df['TotalCharges'].isnull().sum())
print("Min TotalCharges:", df['TotalCharges'].min())

NaN after conversion: 11
TotalCharges dtype: float64
NaN remaining: 0
Min TotalCharges: 0.0


In [3]:
# customerID is just an identifier — not useful for prediction
df.drop('customerID', axis=1, inplace=True)

print("Shape after dropping customerID:", df.shape)
print("Columns remaining:", df.columns.tolist())

Shape after dropping customerID: (7043, 20)
Columns remaining: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [4]:
# Convert Churn from Yes/No to 1/0
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

print("Churn value counts after encoding:")
print(df['Churn'].value_counts())
print("\nChurn dtype:", df['Churn'].dtype)

Churn value counts after encoding:
Churn
0    5174
1    1869
Name: count, dtype: int64

Churn dtype: int64


In [5]:
# Identify categorical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
print("Categorical columns to encode:", cat_cols)
print("Total:", len(cat_cols))

# Binary columns — map directly (Yes/No, Male/Female)
binary_map = {'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0}

binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
               'PaperlessBilling', 'MultipleLines', 'OnlineSecurity',
               'OnlineBackup', 'DeviceProtection', 'TechSupport',
               'StreamingTV', 'StreamingMovies']

for col in binary_cols:
    df[col] = df[col].map(binary_map)
    # Handle 'No internet service' and 'No phone service'
    df[col] = df[col].fillna(0)

print("\nBinary columns encoded successfully")

# Multi-class columns — use one-hot encoding
multi_cols = ['InternetService', 'Contract', 'PaymentMethod']
df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

print("Shape after encoding:", df.shape)
print("\nAll columns now:")
print(df.columns.tolist())

Categorical columns to encode: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Total: 15

Binary columns encoded successfully
Shape after encoding: (7043, 24)

All columns now:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


C:\Users\nandi\AppData\Local\Temp\ipykernel_17660\2323522327.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


In [6]:
# Scale numeric columns so they are on the same range
scaler = StandardScaler()
scale_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

df[scale_cols] = scaler.fit_transform(df[scale_cols])

print("Numeric features scaled successfully")
print("\nSample of scaled values:")
print(df[scale_cols].head(3).round(3))

# Final check
print("\n=== FINAL CLEANED DATASET ===")
print("Shape:", df.shape)
print("Any NaN remaining:", df.isnull().sum().sum())
print("Churn distribution:", df['Churn'].value_counts().to_dict())

Numeric features scaled successfully

Sample of scaled values:
   tenure  MonthlyCharges  TotalCharges
0  -1.277          -1.160        -0.993
1   0.066          -0.260        -0.172
2  -1.237          -0.363        -0.958

=== FINAL CLEANED DATASET ===
Shape: (7043, 24)
Any NaN remaining: 0
Churn distribution: {0: 5174, 1: 1869}


In [7]:
df.to_csv('../data/cleaned_churn.csv', index=False)
print("Cleaned dataset saved to data/cleaned_churn.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {df.shape[1]} features ready for XGBoost")


Cleaned dataset saved to data/cleaned_churn.csv
Shape: (7043, 24)
Columns: 24 features ready for XGBoost
